# Data Quality Scaling Experiment

Compares quality_filtered (QF) vs logged_in_games (LI) at increasing data scales.

- **Exclusive**: trains only on games unique to each dataset, with equal-states (game-level subsampling)
- **Non-exclusive**: trains on overlapping datasets (same games may appear in both)

Capacity schedule: 5K→70L/70T, 10K→100L/100T, 20K→100L/100T, 40K→150L/150T

In [ ]:
import json
import glob
import os
import numpy as np
import matplotlib.pyplot as plt

# Resolve paths relative to this notebook's directory
_NB_DIR = os.path.dirname(os.path.abspath('scaling_plots.ipynb'))

CAPACITY = {
    5000: (70, 70),
    10000: (100, 100),
    20000: (100, 100),
    40000: (150, 150),
    75000: (200, 200),
}

def load_results(pattern, qf_key, li_key):
    """Load scaling results, return list of (max_games, runs) sorted by games."""
    files = sorted(glob.glob(pattern))
    results = []
    for path in files:
        max_games = int(path.split('_')[-1].replace('.json', ''))
        with open(path) as f:
            runs = json.load(f)
        results.append((max_games, runs, qf_key, li_key))
    results.sort(key=lambda x: x[0])
    return results

def extract_metric(results, metric):
    """Extract game counts, means, stds for QF and LI."""
    games, qf_m, qf_s, li_m, li_s, states = [], [], [], [], [], []
    for max_games, runs, qf_key, li_key in results:
        qf_vals = [r[qf_key][metric] for r in runs]
        li_vals = [r[li_key][metric] for r in runs]
        games.append(max_games)
        qf_m.append(np.mean(qf_vals))
        qf_s.append(np.std(qf_vals))
        li_m.append(np.mean(li_vals))
        li_s.append(np.std(li_vals))
        states.append(int(np.mean([r[qf_key]['n_states'] for r in runs])))
    return games, np.array(qf_m), np.array(qf_s), np.array(li_m), np.array(li_s), states

exclusive = load_results('scaling_exclusive_*.json', 'qf_exclusive', 'li_exclusive')
nonexclusive = load_results('scaling_nonexclusive_*.json', 'quality_filtered', 'logged_in_games')

print(f'Exclusive: {len(exclusive)} scale points')
print(f'Non-exclusive: {len(nonexclusive)} scale points')

## Summary Table

In [ ]:
def print_table(results, title):
    print(f'\n{title}')
    print(f'{"Games":>8} {"Cap":>10} {"States":>10}  '
          f'{"QF Loss":>18}  {"LI Loss":>18}  '
          f'{"QF Acc":>18}  {"LI Acc":>18}  '
          f'{"QF Egg":>18}  {"LI Egg":>18}')
    print('-' * 140)
    for max_games, runs, qf_key, li_key in results:
        cap = CAPACITY.get(max_games, ('?', '?'))
        qf_loss = [r[qf_key]['log_loss'] for r in runs]
        li_loss = [r[li_key]['log_loss'] for r in runs]
        qf_acc = [r[qf_key]['accuracy'] for r in runs]
        li_acc = [r[li_key]['accuracy'] for r in runs]
        qf_egg = [r[qf_key]['egg_inversion_rate'] for r in runs]
        li_egg = [r[li_key]['egg_inversion_rate'] for r in runs]
        n_states = int(np.mean([r[qf_key]['n_states'] for r in runs]))
        print(f'{max_games:>8} {cap[0]}L/{cap[1]}T {n_states:>10,}  '
              f'{np.mean(qf_loss):.4f} +/- {np.std(qf_loss):.4f}  '
              f'{np.mean(li_loss):.4f} +/- {np.std(li_loss):.4f}  '
              f'{np.mean(qf_acc):.4f} +/- {np.std(qf_acc):.4f}  '
              f'{np.mean(li_acc):.4f} +/- {np.std(li_acc):.4f}  '
              f'{np.mean(qf_egg):.4f} +/- {np.std(qf_egg):.4f}  '
              f'{np.mean(li_egg):.4f} +/- {np.std(li_egg):.4f}')

print_table(exclusive, 'EXCLUSIVE (equal-states, game-level subsampling)')
print_table(nonexclusive, 'NON-EXCLUSIVE (overlapping datasets)')

## Exclusive Scaling Plots

In [ ]:
def plot_scaling(results, title, filename=None):
    metrics = [
        ('log_loss', 'Log Loss'),
        ('auc_roc', 'AUC-ROC'),
        ('accuracy', 'Accuracy'),
        ('egg_inversion_rate', 'Egg Inversion Rate'),
        ('symmetry_deviation', 'Symmetry Deviation'),
    ]

    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    axes = axes.flatten()

    for ax, (metric_key, metric_label) in zip(axes, metrics):
        games, qf_m, qf_s, li_m, li_s, states = extract_metric(results, metric_key)

        ax.errorbar(games, qf_m, yerr=qf_s,
                    marker='o', capsize=4, label='QF', color='#2196F3')
        ax.errorbar(games, li_m, yerr=li_s,
                    marker='s', capsize=4, label='LI', color='#FF9800')

        ax.set_xlabel('Max Games')
        ax.set_ylabel(metric_label)
        ax.set_title(metric_label)
        ax.set_xscale('log', base=2)
        ax.legend()
        ax.grid(True, alpha=0.3)

        # X-axis labels with capacity
        ax.set_xticks(games)
        labels = []
        for g in games:
            cap = CAPACITY.get(g)
            if cap:
                labels.append(f'{g//1000}K\n{cap[0]}L/{cap[1]}T')
            else:
                labels.append(f'{g//1000}K')
        ax.set_xticklabels(labels, fontsize=8)

    # Hide unused subplot
    for ax in axes[len(metrics):]:
        ax.set_visible(False)

    fig.suptitle(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    if filename:
        plt.savefig(filename, dpi=150, bbox_inches='tight')
        print(f'Saved {filename}')
    plt.show()

if exclusive:
    plot_scaling(exclusive,
                'Exclusive Equal-States Scaling: QF vs LI\n(symmetry-augmented holdout, 10 variance runs)',
                'exclusive_scaling_plot.png')

## Non-Exclusive Scaling Plots

In [ ]:
if nonexclusive:
    plot_scaling(nonexclusive,
                'Non-Exclusive Scaling: QF vs LI\n(symmetry-augmented holdout, 10 variance runs)',
                'nonexclusive_scaling_plot.png')

## Side-by-Side Comparison

Overlay exclusive and non-exclusive results on the same axes to see how dataset overlap affects the comparison.

In [ ]:
if exclusive and nonexclusive:
    metrics = [
        ('log_loss', 'Log Loss'),
        ('auc_roc', 'AUC-ROC'),
        ('accuracy', 'Accuracy'),
        ('egg_inversion_rate', 'Egg Inversion Rate'),
        ('symmetry_deviation', 'Symmetry Deviation'),
    ]

    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    axes = axes.flatten()

    # Find common game counts
    exc_games = set(g for g, _, _, _ in exclusive)
    nex_games = set(g for g, _, _, _ in nonexclusive)
    common = sorted(exc_games & nex_games)

    for ax, (metric_key, metric_label) in zip(axes, metrics):
        eg, eq_m, eq_s, el_m, el_s, _ = extract_metric(exclusive, metric_key)
        ng, nq_m, nq_s, nl_m, nl_s, _ = extract_metric(nonexclusive, metric_key)

        # Slight x-offset to avoid overlapping error bars
        eg_arr = np.array(eg, dtype=float)
        ng_arr = np.array(ng, dtype=float)

        ax.errorbar(eg_arr * 0.95, eq_m, yerr=eq_s,
                    marker='o', capsize=3, label='QF exclusive', color='#1565C0', ls='-')
        ax.errorbar(eg_arr * 1.05, el_m, yerr=el_s,
                    marker='s', capsize=3, label='LI exclusive', color='#E65100', ls='-')
        ax.errorbar(ng_arr * 0.97, nq_m, yerr=nq_s,
                    marker='^', capsize=3, label='QF non-excl', color='#64B5F6', ls='--')
        ax.errorbar(ng_arr * 1.03, nl_m, yerr=nl_s,
                    marker='D', capsize=3, label='LI non-excl', color='#FFB74D', ls='--')

        ax.set_xlabel('Max Games')
        ax.set_ylabel(metric_label)
        ax.set_title(metric_label)
        ax.set_xscale('log', base=2)
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)

        all_games = sorted(exc_games | nex_games)
        ax.set_xticks(all_games)
        ax.set_xticklabels([f'{g//1000}K' for g in all_games], fontsize=8)

    # Hide unused subplot
    for ax in axes[len(metrics):]:
        ax.set_visible(False)

    fig.suptitle('Exclusive vs Non-Exclusive Scaling\n'
                 '(10 variance runs, symmetry-augmented holdout)',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('combined_scaling_plot.png', dpi=150, bbox_inches='tight')
    print('Saved combined_scaling_plot.png')
    plt.show()

## QF Advantage (Delta Plots)

Plot the QF-LI difference at each scale to see if the advantage grows, shrinks, or stays constant.

In [ ]:
def plot_deltas(results, title, filename=None):
    metrics = [
        ('log_loss', 'Log Loss (QF - LI)', -1),       # negative = QF better
        ('auc_roc', 'AUC-ROC (QF - LI)', 1),          # positive = QF better
        ('accuracy', 'Accuracy (QF - LI)', 1),         # positive = QF better
        ('egg_inversion_rate', 'Egg Inversion (QF - LI)', -1),
        ('symmetry_deviation', 'Symmetry Dev (QF - LI)', -1),
    ]

    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    axes = axes.flatten()

    for ax, (metric_key, metric_label, sign) in zip(axes, metrics):
        games_list = []
        delta_means = []
        delta_stds = []

        for max_games, runs, qf_key, li_key in results:
            deltas = [r[qf_key][metric_key] - r[li_key][metric_key] for r in runs]
            games_list.append(max_games)
            delta_means.append(np.mean(deltas))
            delta_stds.append(np.std(deltas))

        delta_means = np.array(delta_means)
        delta_stds = np.array(delta_stds)

        colors = ['#2196F3' if (d * sign > 0) else '#FF5722' for d in delta_means]
        ax.bar(range(len(games_list)), delta_means, yerr=delta_stds,
               capsize=5, color=colors, alpha=0.7, edgecolor='black', linewidth=0.5)
        ax.axhline(y=0, color='black', linewidth=0.8)
        ax.set_xticks(range(len(games_list)))
        ax.set_xticklabels([f'{g//1000}K' for g in games_list])
        ax.set_xlabel('Max Games')
        ax.set_ylabel(metric_label)
        ax.set_title(metric_label)
        ax.grid(True, alpha=0.3, axis='y')

        if sign == -1:
            ax.text(0.98, 0.02, '↓ QF better', transform=ax.transAxes,
                    ha='right', va='bottom', fontsize=8, color='#2196F3')
        else:
            ax.text(0.98, 0.98, '↑ QF better', transform=ax.transAxes,
                    ha='right', va='top', fontsize=8, color='#2196F3')

    # Hide unused subplot
    for ax in axes[len(metrics):]:
        ax.set_visible(False)

    fig.suptitle(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    if filename:
        plt.savefig(filename, dpi=150, bbox_inches='tight')
        print(f'Saved {filename}')
    plt.show()

if exclusive:
    plot_deltas(exclusive,
               'QF Advantage (Exclusive, Equal-States)\nBlue = QF better, Red = LI better',
               'exclusive_delta_plot.png')

In [ ]:
if nonexclusive:
    plot_deltas(nonexclusive,
               'QF Advantage (Non-Exclusive)\nBlue = QF better, Red = LI better',
               'nonexclusive_delta_plot.png')